# TEX 2025 ML 잔차 모델 — 회귀 기반 잔차 설명

> v5_1(타자) 통합본을 슬림화한 ML 잔차 모델 단독 노트북 (2026-05-12)

## 목적

마르코프 시뮬레이션(`integrated_sim.py`, `7.master_v6.ipynb`)이 *within-game 메커니즘*으로 잔차를 측정한다면, ML 잔차 모델은 *season-level stat 회귀*로 잔차를 예측합니다.

- **타겟**: `residual = W - pyth_W` (피타고리안 기대 대비 실제 승수 차이)
- **학습 데이터**: MLB 2015~2024 팀-시즌 (2020 제외, 약 270 행)
- **피처**: `simulator.MODEL_FEATURES` 18개 (불펜·클러치·투구·득점·BB·K 등)
- **모델**: Ridge + Lasso + RF + XGBoost **앙상블 평균**
- **사용처**: `7.master_v6.ipynb` cell 5에서 시뮬 잔차와 cross-check

## 슬림화 이력 (2026-05-12)

원본 `5.통합본(0428+ML 수정)_v5_1(타자).ipynb` (62셀)에서 시뮬·NSGA·Streamlit·Grid 부분 제거 후 ML 모델만 추출. 원본은 `Final/_archive/` 보관.

v5_1에서 시도했던 *타자 7 feature* (wRC_plus, wOBA, BB_pct, K_pct, Hard_pct, ISO_off, GB_pct_off)는 `mlb_team_seasons.csv`에 없어 production `simulator.py`에는 반영되지 않음 (효과 미미).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

for font in ['AppleGothic', 'NanumGothic', 'Apple SD Gothic Neo']:
    if any(font in f.name for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font
        break
plt.rcParams['axes.unicode_minus'] = False

sys.path.insert(0, str(Path.cwd()))
from simulator import (
    _train_model_bundle, _predict_residual, MODEL_FEATURES,
)

## 1. 학습 데이터 + 피처 18개

In [ ]:
FEATURE_DESC = {
    'sv_pct': '세이브 성공률 = SV / (SV + BS)',
    'SV_pg': '경기당 세이브',
    'onerun_wp': '1점차 경기 승률',
    'xi_wp': '연장전 승률',
    'home_away_diff': '홈 승률 − 원정 승률',
    'ERA': '팀 평균자책점',
    'WHIP': '(안타+볼넷) / 이닝',
    'k_bb': 'K/9 ÷ BB/9',
    'K9': '9이닝당 탈삼진',
    'BB9': '9이닝당 볼넷',
    'HR9': '9이닝당 피홈런',
    'ir_pct': '물려받은 주자 실점률 (낮을수록 좋음)',
    'babip_against': '피BABIP',
    'go_ao': '땅볼/플라이볼 비율',
    'OPS': '팀 타선 OPS',
    'rs_per_g': '경기당 득점',
    'sb_pct': '도루 성공률',
    'era_fip_diff': 'ERA − FIP (Fangraphs 피칭 운 보정)',
}

print(f'MODEL_FEATURES: {len(MODEL_FEATURES)}개')
for i, f in enumerate(MODEL_FEATURES, 1):
    print(f'  {i:2d}. {f:18s} — {FEATURE_DESC.get(f, "")}')

raw_dir = Path('data_raw')
hist = pd.read_csv(raw_dir / 'mlb_team_seasons.csv')
hist = hist[(hist['year'] < 2025) & (hist['year'] != 2020)].copy()
hist['pyth_wp'] = hist['RS']**1.83 / (hist['RS']**1.83 + hist['RA']**1.83)
hist['residual'] = hist['W'] - hist['pyth_wp'] * hist['G']

print(f'\n학습 데이터: {len(hist)} 팀-시즌 (2015~2024, 2020 제외)')
print(f'잔차 분포: mean={hist["residual"].mean():+.2f}, std={hist["residual"].std():.2f}')
print(f'잔차 범위: [{hist["residual"].min():+.1f}, {hist["residual"].max():+.1f}]')

## 2. ML 앙상블 학습 + CV 성능

`simulator._train_model_bundle()`이 4모델 (Ridge / Lasso / RF / XGBoost)을 학습하고 GroupKFold (year 단위) CV로 평가합니다. 결과는 디스크 캐시 (`Final/.cache/model_bundle*.pkl`)에 저장되어 재학습 안 함.

In [ ]:
import time
t0 = time.time()
bundle = _train_model_bundle(raw_dir)
print(f'bundle 로드 ({time.time()-t0:.1f}s)')

print(f'\n=== 앙상블 CV 성능 ===')
print(f'  CV MAE 평균: {bundle.ensemble_cv_mae:.3f}승')
print(f'  (잔차 std {hist["residual"].std():.2f}승 대비 → CV MAE가 std의 {bundle.ensemble_cv_mae/hist["residual"].std()*100:.0f}%)')

print(f'\n=== 학습된 모델 ===')
print(f'  Ridge   α = {bundle.ridge.alpha:.3f}')
print(f'  Lasso   α = {bundle.lasso.alpha:.3f}')
print(f'  RF      n_estimators=300, min_samples_leaf=5')
print(f'  XGBoost n_estimators=100, learning_rate=0.03, max_depth=2')

## 3. 피처 중요도 — 어느 피처가 잔차를 설명하나

In [ ]:
from sklearn.inspection import permutation_importance

# 학습 데이터 재구성 (importance 계산용)
hist['k_bb'] = hist['K9'] / hist['BB9'].clip(lower=0.1)
hist['rs_per_g'] = hist['RS'] / hist['G']
hist['SV_pg'] = hist['SV'] / hist['G']
hist['home_away_diff'] = hist['home_wp'] - hist['away_wp']
df_model = hist.dropna(subset=MODEL_FEATURES + ['residual']).copy()
X_raw = df_model[MODEL_FEATURES].values
y = df_model['residual'].values
X_sc = bundle.scaler.transform(X_raw)

ridge_imp = np.abs(bundle.ridge.coef_) / (np.abs(bundle.ridge.coef_).sum() + 1e-9)
lasso_imp = np.abs(bundle.lasso.coef_) / (np.abs(bundle.lasso.coef_).sum() + 1e-9)
pi = permutation_importance(bundle.rf, X_raw, y, n_repeats=30, random_state=42)
rf_imp = pi.importances_mean.clip(0) / (pi.importances_mean.clip(0).sum() + 1e-9)
xgb_imp = bundle.boost_model.feature_importances_ / (bundle.boost_model.feature_importances_.sum() + 1e-9)

imp_df = pd.DataFrame({
    'feature': MODEL_FEATURES,
    'Ridge': ridge_imp, 'Lasso': lasso_imp,
    'RF': rf_imp, 'XGBoost': xgb_imp,
})
imp_df['ensemble'] = imp_df[['Ridge','Lasso','RF','XGBoost']].mean(axis=1)
imp_df = imp_df.sort_values('ensemble', ascending=False).reset_index(drop=True)

print('=== 피처 중요도 (앙상블 평균 기준 top 8) ===')
print(imp_df.head(8).to_string(index=False, float_format='%.3f'))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(f'잔차 모델 피처 중요도 — CV MAE {bundle.ensemble_cv_mae:.2f}승', fontweight='bold')

feat_order = imp_df['feature'].tolist()
w = 0.2
colors4 = ['#4E79A7', '#F28E2B', '#59A14F', '#E15759']
ax = axes[0]
for i, (col, color) in enumerate(zip(['Ridge','Lasso','RF','XGBoost'], colors4)):
    vals = [imp_df.loc[imp_df['feature']==f, col].values[0] for f in feat_order]
    ax.barh(np.arange(len(feat_order)) + i*w, vals, w, label=col, color=color, alpha=0.85)
ax.set_yticks(np.arange(len(feat_order)) + 1.5*w)
ax.set_yticklabels(feat_order)
ax.invert_yaxis()
ax.set_xlabel('정규화 중요도')
ax.set_title('모델별 중요도')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
isorted = imp_df.sort_values('ensemble', ascending=True)
ax.barh(isorted['feature'], isorted['ensemble'], color='#76B7B2', alpha=0.85)
ax.set_xlabel('앙상블 평균 중요도')
ax.set_title('앙상블 평균')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 4. TEX 2025 잔차 예측

학습된 앙상블로 TEX 2025의 stat을 입력해 *얼마나 잔차를 설명할 수 있는지* 측정합니다.

In [ ]:
tex25 = bundle.tex25
ml_resid = _predict_residual(bundle, tex25)
actual_resid = tex25['W'] - tex25['pyth_W']

print(f'=== TEX 2025 잔차 예측 ===')
print(f'  실제 W            : {tex25["W"]}')
print(f'  피타고리안 W      : {tex25["pyth_W"]:.2f}')
print(f'  실제 잔차         : {actual_resid:+.2f}')
print(f'  ML 앙상블 예측 잔차: {ml_resid:+.2f}')
print(f'  예측 오차         : {ml_resid - actual_resid:+.2f}')
print()
print(f'  CV MAE 기준 1σ 범위: ±{bundle.ensemble_cv_mae:.2f}승')
print(f'  → ML 예측이 CV MAE 범위 안에 있으면 *신뢰 가능*, 밖이면 *외삽 영역*')

## 5. 마스터 노트북에서의 활용

이 ML 모델은 `7.master_v6.ipynb`에서 두 가지로 활용:

1. **잔차 cross-check** (cell 5): 시뮬 baseline 잔차(-0.2) vs ML 예측 잔차 비교 — 두 도구가 잔차를 어디까지 설명하는지
2. **외삽 한계 시각화** (cell 12-14): NSGA-II Phase 8 σ 시나리오가 ML 학습 분포(270 팀-시즌) *밖*으로 가는지 확인. 분포 밖 σ는 ML 신뢰 X, 시뮬만 신뢰

### v5_1 → v5 (현재) 차이

| | v5_1(타자) 원본 (62셀) | 현재 슬림본 (8셀) |
|---|---|---|
| 시뮬 코드 | 포함 | 제거 (v6 마스터로 이동) |
| NSGA/Pareto | 포함 | 제거 (v6 마스터로 이동) |
| Streamlit/Grid | 포함 | 제거 |
| ML 학습 코드 | inline (cell 25-26) | `simulator._train_model_bundle` 호출 |
| 피처 | 25개 (타자 7 추가 시도) | 18개 (production 일관) |

타자 7 feature (wRC_plus 등)는 v5_1에서 시도했으나 `mlb_team_seasons.csv`에 없어 production에 반영 안 됨. 효과 미미한 것으로 평가.